In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import re
import string
import warnings
from collections import Counter
 
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 130, "axes.spines.top": False, "axes.spines.right": False})

LOAD DATASET

In [ ]:
df = pd.read_csv("data/raw/trainingdata.csv")
print("Dataset loaded.\n")

DATASET OVERVIEW

In [ ]:
print("=" * 60)
print("STEP 1 — DATASET OVERVIEW")
print("=" * 60)
 
print(f"\nShape         : {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Columns       : {list(df.columns)}")
print(f"\nData types:\n{df.dtypes}")
 
print("\n--- Memory usage ---")
df.info(memory_usage="deep")
 
print("\n--- First 5 rows ---")
print(df.head())
 
print("\n--- Last 5 rows ---")
print(df.tail())
 
print("\n--- 5 random samples ---")
print(df.sample(5, random_state=42))

LABEL DISTRIBUTION

In [ ]:
print("\n" + "=" * 60)
print("STEP 2 — LABEL DISTRIBUTION")
print("=" * 60)
 
counts = df["Label"].value_counts()
pcts   = df["Label"].value_counts(normalize=True) * 100
 
print(f"\n{'Label':<10} {'Count':>10} {'Percentage':>12}")
print("-" * 34)
for label in [0, 1]:
    name = "Benign (0)" if label == 0 else "SQLi   (1)"
    print(f"{name:<10} {counts[label]:>10,} {pcts[label]:>11.2f}%")
 
imbalance_ratio = counts[1] / counts[0]
print(f"\nImbalance ratio (SQLi:Benign) : {imbalance_ratio:.2f}")
if imbalance_ratio > 2:
    print("⚠  Significant imbalance — consider SMOTE or class_weight='balanced'")
else:
    print("✓  Mild imbalance — class_weight='balanced' should be sufficient")
 
fig, ax = plt.subplots(figsize=(5, 4))
bars = ax.bar(["Benign (0)", "SQLi (1)"], counts.values,
              color=["#4A90D9", "#E05C5C"], width=0.5, edgecolor="white")
for bar, pct in zip(bars, pcts.values):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 400, f"{pct:.1f}%", ha="center", fontsize=11, fontweight="bold")
ax.set_title("Label distribution", fontsize=13, fontweight="bold")
ax.set_ylabel("Count")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
plt.tight_layout()
plt.savefig("step2_label_distribution.png")
plt.show()
print("→ Saved: step2_label_distribution.png")

MISSING VALUES & DUPLICATES

In [ ]:
print("\n" + "=" * 60)
print("STEP 3 — MISSING VALUES & DUPLICATES")
print("=" * 60)
 
print("\n--- Null counts ---")
print(df.isnull().sum())
 
total_dups = df.duplicated().sum()
print(f"\nFully duplicate rows  : {total_dups:,}")
 
# Queries appearing in BOTH classes (potential label leakage)
shared = df.groupby("Query")["Label"].nunique()
leaked = shared[shared > 1]
print(f"Queries in both classes (label leakage) : {len(leaked):,}")
if len(leaked):
    print("Sample leaked queries:")
    print(df[df["Query"].isin(leaked.index)].head(6))
else:
    print("✓  No label leakage detected")

QUERY LENGHT ANALYSIS

In [ ]:
print("\n" + "=" * 60)
print("STEP 4 — QUERY LENGTH ANALYSIS")
print("=" * 60)
 
df["char_len"]  = df["Query"].str.len()
df["word_count"] = df["Query"].str.split().str.len()
 
for col, label in [("char_len", "Character length"), ("word_count", "Word count")]:
    print(f"\n--- {label} ---")
    print(df.groupby("Label")[col].describe().round(2).to_string())
    p95 = df[col].quantile(0.95)
    p99 = df[col].quantile(0.99)
    print(f"  p95 = {p95:.0f}    p99 = {p99:.0f}")
 
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, col, title in zip(axes,
                           ["char_len", "word_count"],
                           ["Character length", "Word count"]):
    for label, color, name in [(0, "#4A90D9", "Benign"), (1, "#E05C5C", "SQLi")]:
        subset = df[df["Label"] == label][col].clip(upper=df[col].quantile(0.99))
        ax.hist(subset, bins=50, alpha=0.55, color=color, label=name, edgecolor="none")
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel(col)
    ax.set_ylabel("Frequency")
    ax.legend()
plt.suptitle("Query length distributions by class", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("step4_query_lengths.png")
plt.show()
print("→ Saved: step4_query_lengths.png")

CHARACTER AND ENCODING CHECKS

In [ ]:
print("\n" + "=" * 60)
print("STEP 5 — CHARACTER & ENCODING CHECKS")
print("=" * 60)
 
df["has_non_ascii"]  = df["Query"].apply(lambda q: bool(re.search(r"[^\x00-\x7F]", str(q))))
df["has_html_enc"]   = df["Query"].apply(lambda q: bool(re.search(r"&[a-z]+;|&#\d+;|%[0-9a-fA-F]{2}", str(q))))
df["has_null_byte"]  = df["Query"].apply(lambda q: "\x00" in str(q))
df["has_ctrl_chars"] = df["Query"].apply(lambda q: bool(re.search(r"[\x01-\x1f\x7f]", str(q))))
df["has_multi_ws"]   = df["Query"].apply(lambda q: "  " in str(q))
 
checks = ["has_non_ascii", "has_html_enc", "has_null_byte", "has_ctrl_chars", "has_multi_ws"]
check_labels = ["Non-ASCII chars", "HTML/URL encoding", "Null bytes",
                "Control characters", "Multiple whitespace"]
 
print(f"\n{'Check':<25} {'Total':>8} {'% of data':>10}")
print("-" * 45)
for col, lbl in zip(checks, check_labels):
    total = df[col].sum()
    pct   = total / len(df) * 100
    print(f"{lbl:<25} {total:>8,} {pct:>9.2f}%")
 
print("\n--- Breakdown by class ---")
print(df.groupby("Label")[checks].sum().to_string())
 
# Samples with encoding anomalies
html_samples = df[df["has_html_enc"]]["Query"].head(5)
if len(html_samples):
    print("\nSample HTML/URL-encoded queries:")
    for q in html_samples:
        print(f"  {q[:120]}")

SQL KEYWORD FREQUENCY

In [ ]:
print("\n" + "=" * 60)
print("STEP 6 — SQL KEYWORD FREQUENCY")
print("=" * 60)
 
SQL_KEYWORDS = [
    "select", "union", "insert", "update", "delete", "drop", "create",
    "alter", "exec", "execute", "from", "where", "or", "and", "not",
    "null", "like", "order", "group", "having", "join", "cast", "convert",
    "--", "/*", "*/", "xp_", "0x", "char(", "version(", "sleep(",
]
 
def count_keyword(text, kw):
    return str(text).lower().count(kw)
 
kw_stats = {}
for kw in SQL_KEYWORDS:
    col = f"kw_{kw.replace(' ', '_').replace('(', '').replace('/', '_')}"
    df[col] = df["Query"].apply(lambda q, k=kw: count_keyword(q, k))
    kw_stats[kw] = {
        "benign_mean": df[df["Label"] == 0][col].mean(),
        "sqli_mean":   df[df["Label"] == 1][col].mean(),
    }
 
kw_df = pd.DataFrame(kw_stats).T
kw_df["ratio"] = (kw_df["sqli_mean"] + 1e-9) / (kw_df["benign_mean"] + 1e-9)
kw_df = kw_df.sort_values("ratio", ascending=False)
 
print(f"\n{'Keyword':<15} {'Benign mean':>13} {'SQLi mean':>11} {'SQLi/Benign':>13}")
print("-" * 55)
for kw, row in kw_df.iterrows():
    print(f"{kw:<15} {row['benign_mean']:>13.4f} {row['sqli_mean']:>11.4f} {row['ratio']:>13.2f}x")
 
top_kws = kw_df.head(12)
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(top_kws))
w = 0.38
ax.bar(x - w/2, top_kws["benign_mean"], w, label="Benign", color="#4A90D9", alpha=0.85)
ax.bar(x + w/2, top_kws["sqli_mean"],   w, label="SQLi",   color="#E05C5C", alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(top_kws.index, rotation=40, ha="right", fontsize=9)
ax.set_ylabel("Mean count per query")
ax.set_title("SQL keyword frequency — top 12 by SQLi/Benign ratio", fontweight="bold")
ax.legend()
plt.tight_layout()
plt.savefig("step6_keyword_frequency.png")
plt.show()
print("→ Saved: step6_keyword_frequency.png")

SPECIAL CHARACTER ANALYSIS

In [ ]:
print("\n" + "=" * 60)
print("STEP 7 — SPECIAL CHARACTER ANALYSIS")
print("=" * 60)
 
SPECIAL_CHARS = {
    "single_quote":   "'",
    "double_quote":   '"',
    "semicolon":      ";",
    "dash_dash":      "--",
    "hash":           "#",
    "open_paren":     "(",
    "close_paren":    ")",
    "open_comment":   "/*",
    "close_comment":  "*/",
    "equals":         "=",
    "backslash":      "\\",
    "pipe":           "|",
}
 
for name, ch in SPECIAL_CHARS.items():
    df[f"sc_{name}"] = df["Query"].apply(lambda q, c=ch: str(q).count(c))
 
sc_cols = [f"sc_{n}" for n in SPECIAL_CHARS]
sc_summary = df.groupby("Label")[sc_cols].mean().T
sc_summary.columns = ["Benign mean", "SQLi mean"]
sc_summary.index = list(SPECIAL_CHARS.keys())
sc_summary["ratio"] = (sc_summary["SQLi mean"] + 1e-9) / (sc_summary["Benign mean"] + 1e-9)
sc_summary = sc_summary.sort_values("ratio", ascending=False)
 
print(f"\n{'Character':<18} {'Benign mean':>13} {'SQLi mean':>11} {'Ratio':>8}")
print("-" * 54)
for idx, row in sc_summary.iterrows():
    print(f"{idx:<18} {row['Benign mean']:>13.4f} {row['SQLi mean']:>11.4f} {row['ratio']:>8.2f}x")
 
fig, ax = plt.subplots(figsize=(10, 4))
ax.barh(sc_summary.index, sc_summary["ratio"], color="#7B61FF", alpha=0.8)
ax.axvline(1, color="gray", linestyle="--", linewidth=0.8)
ax.set_xlabel("SQLi / Benign mean ratio")
ax.set_title("Special character enrichment in SQLi vs benign queries", fontweight="bold")
plt.tight_layout()
plt.savefig("step7_special_chars.png")
plt.show()
print("→ Saved: step7_special_chars.png")

SAMPLE INSPECTION BY CLASS

In [ ]:
print("\n" + "=" * 60)
print("STEP 8 — SAMPLE INSPECTION BY CLASS")
print("=" * 60)
 
for label, name in [(0, "Benign"), (1, "SQLi")]:
    print(f"\n--- {name} (Label={label}) — 10 random samples ---")
    samples = df[df["Label"] == label]["Query"].sample(10, random_state=7).values
    for i, q in enumerate(samples, 1):
        print(f"  {i:>2}. {str(q)[:140]}")
 
# Obfuscation pattern check
obfuscation_patterns = {
    "mixed case SELECT": r"(?i)s[Ee][Ll][Ee][Cc][Tt]",
    "inline comment":    r"/\*.*?\*/",
    "char() function":   r"(?i)char\s*\(",
    "hex encoding":      r"0x[0-9a-fA-F]+",
    "URL encoding":      r"%[0-9a-fA-F]{2}",
}
 
print("\n--- Obfuscation pattern counts (SQLi class only) ---")
sqli_df = df[df["Label"] == 1]
for name, pattern in obfuscation_patterns.items():
    count = sqli_df["Query"].str.contains(pattern, regex=True, na=False).sum()
    pct   = count / len(sqli_df) * 100
    print(f"  {name:<25} : {count:,} ({pct:.1f}% of SQLi samples)")

VOCABULARY OVERLAP

In [ ]:
print("\n" + "=" * 60)
print("STEP 9 — VOCABULARY OVERLAP")
print("=" * 60)
 
def tokenize(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9_ ]", " ", text)
    return text.split()
 
benign_tokens = Counter()
sqli_tokens   = Counter()
 
for query, label in zip(df["Query"], df["Label"]):
    tokens = tokenize(query)
    if label == 0:
        benign_tokens.update(tokens)
    else:
        sqli_tokens.update(tokens)
 
benign_vocab = set(benign_tokens.keys())
sqli_vocab   = set(sqli_tokens.keys())
shared_vocab = benign_vocab & sqli_vocab
 
print(f"\nBenign vocabulary size  : {len(benign_vocab):,}")
print(f"SQLi vocabulary size    : {len(sqli_vocab):,}")
print(f"Shared tokens           : {len(shared_vocab):,}")
print(f"Exclusive to benign     : {len(benign_vocab - sqli_vocab):,}")
print(f"Exclusive to SQLi       : {len(sqli_vocab - benign_vocab):,}")
 
# Tokens exclusive to SQLi (top 30 by frequency)
sqli_exclusive = {t: c for t, c in sqli_tokens.items() if t not in benign_vocab}
print(f"\nTop 20 tokens exclusive to SQLi:")
for tok, cnt in sorted(sqli_exclusive.items(), key=lambda x: -x[1])[:20]:
    print(f"  {tok:<25} count: {cnt:,}")
 
# Suggest max_features for TF-IDF
total_vocab = len(benign_vocab | sqli_vocab)
suggested_features = min(10_000, total_vocab)
print(f"\nTotal vocabulary        : {total_vocab:,}")
print(f"Suggested TF-IDF max_features : {suggested_features:,}")

BASELINE SEPARABILITY CHECK

In [ ]:
print("\n" + "=" * 60)
print("STEP 10 — BASELINE SEPARABILITY CHECK")
print("=" * 60)
 
X = df["Query"].fillna("")
y = df["Label"]
 
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
 
tfidf = TfidfVectorizer(max_features=10_000, ngram_range=(1, 2), sublinear_tf=True)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)
 
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42),
    "Naive Bayes":         MultinomialNB(alpha=0.1),
}
 
for name, model in models.items():
    model.fit(X_train_tfidf, y_train)
    y_pred = model.predict(X_test_tfidf)
    y_prob = model.predict_proba(X_test_tfidf)[:, 1]
    auc = roc_auc_score(y_test, y_prob)
    print(f"\n--- {name} ---")
    print(classification_report(y_test, y_pred, target_names=["Benign", "SQLi"]))
    print(f"ROC-AUC : {auc:.4f}")
 
print("\n" + "=" * 60)
print("DATA EXPLORATION COMPLETE")
print("=" * 60)

print("""
Summary columns added to df:
  char_len, word_count          — length features
  has_non_ascii, has_html_enc,
  has_null_byte, has_ctrl_chars,
  has_multi_ws                  — encoding flags
  kw_*                          — SQL keyword counts
  sc_*                          — special character counts
 
These columns feed directly into Step 3 (feature engineering).
""")
